# RDF Benchmark Comparison: 8 Synthetic Data Generators

This notebook executes **eight** RDF synthetic data generators and compares their performance and characteristics.

## Overview

- **BSBM**: E-commerce domain with products, vendors, offers, and reviews
- **LUBM**: University domain with departments, professors, students, and courses  
- **GAIA**: Ontology instance generator using LUBM ontology for university domain data
- **LINKGEN**: Flexible synthetic linked data generator with configurable distributions (Zipf/Gaussian)
- **PyGraft**: Configurable schema and knowledge graph generator with RDFS/OWL constructs
- **RDFMutate**: Graph mutation-based synthetic data generator using SWRL/SHACL operators
- **RDFGraphGen**: SHACL-based synthetic data generator that generates RDF from shape definitions
- **RUDOF Generate**: High-performance RDF generator using ShEx schemas with configurable cardinality strategies

In [3]:
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Configuration

Set the parameters for both benchmarks:

In [1]:
# BSBM Configuration
BSBM_PRODUCTS = 10000  # Number of products to generate
BSBM_FORMAT = "ttl"    # Output format: ttl, nt, n3, trig

# LUBM Configuration
LUBM_UNIVERSITIES = 10  # Number of universities to generate
LUBM_SEED = 0          # Random seed for reproducibility

# GAIA Configuration - Increased dataset size
GAIA_INSTANCES = 1000     # Number of instances per class (increased from 5 to 20)
GAIA_MATERIALIZATION = True  # Enable materialization

# LINKGEN Configuration
LINKGEN_TRIPLES = 2000000     # Number of triples to generate
LINKGEN_ONTOLOGY = "dbpedia_2015.owl"  # Ontology: dbpedia_2015.owl or schemaorg.owl
LINKGEN_DISTRIBUTION = "zipf"  # Distribution: zipf or gaussian
LINKGEN_THREADS = 4            # Number of threads

# PyGraft Configuration
PYGRAFT_MODE = "full"          # Mode: schema, kg, or full
PYGRAFT_CLASSES = 30           # Number of classes in schema
PYGRAFT_RELATIONS = 20         # Number of relations/properties
PYGRAFT_AVG_INSTANCES = 80     # Average instances per class
PYGRAFT_CHECK_CONSISTENCY = False  # Enable HermiT reasoner (slower but ensures consistency)

# RDFMutate Configuration
RDFMUTATE_MUTATIONS = 100      # Number of mutations to apply per graph
RDFMUTATE_MUTANTS = 10         # Number of mutant graphs to generate
RDFMUTATE_CONFIG = "config.yaml"  # Base configuration file path

# RDFGraphGen Configuration
RDFGRAPHGEN_SCALE_FACTOR = 100  # Scale factor for graph generation (controls size)
RDFGRAPHGEN_SHAPE = "input-shape.ttl"  # SHACL shape file
RDFGRAPHGEN_OUTPUT = "output-graph.ttl"  # Output file name

# RUDOF Generate Configuration (Binary v0.1.142)
RUDOFGENERATE_ENTITY_COUNT = 100000  # Number of entities to generate
RUDOFGENERATE_SCHEMA = "example_schema.shex"  # ShEx schema file
RUDOFGENERATE_OUTPUT_FORMAT = "turtle"  # Output format: turtle, ntriples, rdfxml, trig, n3, nquads, jsonld
RUDOFGENERATE_SEED = 42  # Random seed for reproducibility
RUDOFGENERATE_PARALLEL_THREADS = 8  # Number of parallel threads for processing

print(f"BSBM Configuration: {BSBM_PRODUCTS:,} products, format: {BSBM_FORMAT}")
print(f"LUBM Configuration: {LUBM_UNIVERSITIES} universities, seed: {LUBM_SEED}")
print(f"GAIA Configuration: {GAIA_INSTANCES} instances per class, materialization: {GAIA_MATERIALIZATION}")
print(f"LINKGEN Configuration: {LINKGEN_TRIPLES:,} triples, ontology: {LINKGEN_ONTOLOGY}, distribution: {LINKGEN_DISTRIBUTION}")
print(f"PyGraft Configuration: {PYGRAFT_MODE} mode, {PYGRAFT_CLASSES} classes, {PYGRAFT_AVG_INSTANCES} avg instances/class")
print(f"RDFMutate Configuration: {RDFMUTATE_MUTATIONS} mutations × {RDFMUTATE_MUTANTS} mutants = ~{RDFMUTATE_MUTATIONS * RDFMUTATE_MUTANTS} total operations")
print(f"RDFGraphGen Configuration: scale factor: {RDFGRAPHGEN_SCALE_FACTOR}, shape: {RDFGRAPHGEN_SHAPE}")
print(f"RUDOF Generate Configuration (Binary): {RUDOFGENERATE_ENTITY_COUNT} entities, format: {RUDOFGENERATE_OUTPUT_FORMAT}, seed: {RUDOFGENERATE_SEED}, threads: {RUDOFGENERATE_PARALLEL_THREADS}")

BSBM Configuration: 10,000 products, format: ttl
LUBM Configuration: 10 universities, seed: 0
GAIA Configuration: 1000 instances per class, materialization: True
LINKGEN Configuration: 2,000,000 triples, ontology: dbpedia_2015.owl, distribution: zipf
PyGraft Configuration: full mode, 30 classes, 80 avg instances/class
RDFMutate Configuration: 100 mutations × 10 mutants = ~1000 total operations
RDFGraphGen Configuration: scale factor: 100, shape: input-shape.ttl
RUDOF Generate Configuration (Binary): 100000 entities, format: turtle, seed: 42, threads: 8


## 1. Execute BSBM Benchmark

In [5]:
print("=" * 80)
print("EXECUTING BSBM BENCHMARK")
print("=" * 80)

bsbm_cmd = [
    "python3",
    "BSBM/execute_benchmark.py",
    "--products", str(BSBM_PRODUCTS),
    "--format", BSBM_FORMAT
]

print(f"Running command: {' '.join(bsbm_cmd)}\n")

bsbm_result = subprocess.run(
    bsbm_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(bsbm_result.stdout)
if bsbm_result.returncode != 0:
    print("ERROR:", bsbm_result.stderr)
else:
    print("✅ BSBM benchmark completed successfully!")

EXECUTING BSBM BENCHMARK
Running command: python3 BSBM/execute_benchmark.py --products 10000 --format ttl

Running BSBM Generator...
Command: java -cp bsbm.jar:ssj.jar -Xmx2G benchmark.generator.Generator -pc 10000 -s ttl -fn output/dataset -fc

✅ BSBM Generation Complete!
⏱️  Execution Time: 7.91 seconds
📦 Products Generated: 10000
🔢 Total Triples: 3,564,773
⚡ Triples/sec: 450,564
📁 Output Format: TTL
📂 Output Location: /home/diego/Documents/benchmarks-synthetic-data-generators/BSBM/output/dataset.ttl

📊 Detailed report saved to: /home/diego/Documents/benchmarks-synthetic-data-generators/BSBM/output/benchmark_report.json

✅ BSBM benchmark completed successfully!
Running BSBM Generator...
Command: java -cp bsbm.jar:ssj.jar -Xmx2G benchmark.generator.Generator -pc 10000 -s ttl -fn output/dataset -fc

✅ BSBM Generation Complete!
⏱️  Execution Time: 7.91 seconds
📦 Products Generated: 10000
🔢 Total Triples: 3,564,773
⚡ Triples/sec: 450,564
📁 Output Format: TTL
📂 Output Location: /home/dieg

## 2. Execute LUBM Benchmark

In [6]:
print("=" * 80)
print("EXECUTING LUBM BENCHMARK")
print("=" * 80)

lubm_cmd = [
    "python3",
    "LUBM/execute_benchmark.py",
    "--universities", str(LUBM_UNIVERSITIES),
    "--seed", str(LUBM_SEED)
]

print(f"Running command: {' '.join(lubm_cmd)}\n")

lubm_result = subprocess.run(
    lubm_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(lubm_result.stdout)
if lubm_result.returncode != 0:
    print("ERROR:", lubm_result.stderr)
else:
    print("✅ LUBM benchmark completed successfully!")

EXECUTING LUBM BENCHMARK
Running command: python3 LUBM/execute_benchmark.py --universities 10 --seed 0

LUBM Generator Executor
Configuration:
  Universities: 10
  Starting index: 0
  Random seed: 0
  Ontology URL: http://www.lehigh.edu/~yug2/Research/SemanticWeb/LUBM/univ-bench.owl
  Output format: owl

Executing LUBM generator with command:
java -cp /home/diego/Documents/benchmarks-synthetic-data-generators/LUBM/lubm-generator-fixed.jar edu.lehigh.swat.bench.uba.Generator -univ 10 -index 0 -seed 0 -onto http://www.lehigh.edu/~yug2/Research/SemanticWeb/LUBM/univ-bench.owl

LUBM generator executed successfully in 4.335779 seconds (4335.779ms)!
Output:
Started...
/home/diego/Documents/benchmarks-synthetic-data-generators/LUBM/output/University0_0.owl generated
CLASS INSTANCE #: 1657, TOTAL SO FAR: 1657
PROPERTY INSTANCE #: 6896, TOTAL SO FAR: 6896

/home/diego/Documents/benchmarks-synthetic-data-generators/LUBM/output/University0_1.owl generated
CLASS INSTANCE #: 1327, TOTAL SO FAR: 298

## 3. Execute GAIA Benchmark

In [5]:
print("=" * 80)
print("EXECUTING GAIA BENCHMARK")
print("=" * 80)

gaia_cmd = [
    "python3",
    "execute_benchmark.py",  # Run from GAIA directory
    "--instances", str(GAIA_INSTANCES)
]

if GAIA_MATERIALIZATION:
    gaia_cmd.append("--materialization")

print(f"Running command: {' '.join(gaia_cmd)} (in GAIA directory)\n")

gaia_result = subprocess.run(
    gaia_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd() / "GAIA"  # Change working directory to GAIA
)

print(gaia_result.stdout)
if gaia_result.returncode != 0:
    print("ERROR:", gaia_result.stderr)
else:
    print("✅ GAIA benchmark completed successfully!")

EXECUTING GAIA BENCHMARK
Running command: python3 execute_benchmark.py --instances 1000 --materialization (in GAIA directory)

🎯 GAIA Generator with LUBM Ontology
✅ LUBM ontology found: univ-bench.owl
🚀 Running GAIA Generator...
📁 Input ontology: univ-bench.owl
📁 Output file: output/gaia_instances.owl
🔢 Instances per class: 1000
⚡ Materialization: Yes
💻 Command: java -Xmx8g -jar OWLGenerator.jar -F univ-bench.owl -O output/gaia_instances.owl -N 1000 -M
✅ GAIA generation completed successfully!

📊 Benchmark Results:
⏱️  Execution Time: 0:00:00
🔢 Instances Generated: 0
📁 Output Size: 15.68 MB
📄 Report saved to: output/benchmark_report.json
💾 Output saved to: output/gaia_instances.owl

✅ GAIA benchmark completed successfully!


## 4. Execute LINKGEN Benchmark

In [7]:
print("=" * 80)
print("EXECUTING LINKGEN BENCHMARK")
print("=" * 80)

linkgen_cmd = [
    "python3",
    "LINKGEN/execute_benchmark.py",
    "--triples", str(LINKGEN_TRIPLES),
    "--ontology", LINKGEN_ONTOLOGY,
    "--distribution", LINKGEN_DISTRIBUTION,
    "--threads", str(LINKGEN_THREADS)
]

print(f"Running command: {' '.join(linkgen_cmd)}\n")

linkgen_result = subprocess.run(
    linkgen_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(linkgen_result.stdout)
if linkgen_result.returncode != 0:
    print("ERROR:", linkgen_result.stderr)
else:
    print("✅ LINKGEN benchmark completed successfully!")

EXECUTING LINKGEN BENCHMARK
Running command: python3 LINKGEN/execute_benchmark.py --triples 2000000 --ontology dbpedia_2015.owl --distribution zipf --threads 4

Running LINKGEN Generator...
Ontology: dbpedia_2015.owl
Triples: 2,000,000
Distribution: zipf
Threads: 4

✅ LINKGEN Generation Complete!
⏱️  Execution Time: 99.59 seconds
🔢 Triples Requested: 2,000,000
📊 Files Generated: 6
💾 Total Size: 2650.64 MB
⚡ Throughput: 20,082 triples/sec
📁 Output Format: NT
📂 Output Location: /home/diego/Documents/benchmarks-synthetic-data-generators/LINKGEN/output

📊 Detailed report saved to: /home/diego/Documents/benchmarks-synthetic-data-generators/LINKGEN/output/benchmark_report.json

✅ LINKGEN benchmark completed successfully!
Running LINKGEN Generator...
Ontology: dbpedia_2015.owl
Triples: 2,000,000
Distribution: zipf
Threads: 4

✅ LINKGEN Generation Complete!
⏱️  Execution Time: 99.59 seconds
🔢 Triples Requested: 2,000,000
📊 Files Generated: 6
💾 Total Size: 2650.64 MB
⚡ Throughput: 20,082 triple

## 5. Execute PyGraft Benchmark

In [7]:
print("=" * 80)
print("EXECUTING PYGRAFT BENCHMARK")
print("=" * 80)

pygraft_cmd = [
    "docker", "run", "--rm",
    "-v", f"{Path.cwd()}/PYGRAFT/output:/app/output",
    "pygraft-benchmark:latest",
    "python", "execute_benchmark.py",
    "--mode", PYGRAFT_MODE,
    "--n-classes", str(PYGRAFT_CLASSES),
    "--n-relations", str(PYGRAFT_RELATIONS),
    "--avg-instances", str(PYGRAFT_AVG_INSTANCES)
]

if not PYGRAFT_CHECK_CONSISTENCY:
    pygraft_cmd.append("--no-check-consistency")

print(f"Running command: {' '.join(pygraft_cmd)}\n")

pygraft_result = subprocess.run(
    pygraft_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(pygraft_result.stdout)
if pygraft_result.returncode != 0:
    print("ERROR:", pygraft_result.stderr)
else:
    print("✅ PyGraft benchmark completed successfully!")

EXECUTING PYGRAFT BENCHMARK
Running command: docker run --rm -v /home/diego/Documents/benchmarks-synthetic-data-generators/PYGRAFT/output:/app/output pygraft-benchmark:latest python execute_benchmark.py --mode full --n-classes 30 --n-relations 20 --avg-instances 80 --no-check-consistency

🎯 PyGraft - Knowledge Graph Generator
Configuration:
  Mode: full
  Classes: 30
  Relations: 20
  Avg Instances/Class: 80
  Output Directory: /app/output
  Check Consistency: False
  Multitask: False
  Random Seed: 42

📝 Creating configuration file...
   Config saved to: /app/config.yml

🚀 Starting PyGraft generation (full mode)...
   This may take a few minutes depending on size...



  ____      __   __    ____      ____         _        _____    _____   
U|  _"\ u   \ \ / / U /"___|u U |  _"\ u  U  /"\  u   |" ___|  |_ " _|  
\| |_) |/    \ V /  \| |  _ /  \| |_) |/   \/ _ \/   U| |_  u    | |    
 |  __/     U_|"|_u  | |_| |    |  _ <     / ___ \   \|  _|/    /| |\   
 |_|          |_|     \____| 

## 6. Execute RDFMutate Benchmark

In [8]:
print("=" * 80)
print("EXECUTING RDFMUTATE BENCHMARK")
print("=" * 80)

rdfmutate_cmd = [
    "python3",
    "RDFMUTATE/execute_benchmark.py",
    "--config", RDFMUTATE_CONFIG
]

print(f"Running command: {' '.join(rdfmutate_cmd)}\n")

rdfmutate_result = subprocess.run(
    rdfmutate_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(rdfmutate_result.stdout)
if rdfmutate_result.returncode != 0:
    print("ERROR:", rdfmutate_result.stderr)
else:
    print("✅ RDFMutate benchmark completed successfully!")

EXECUTING RDFMUTATE BENCHMARK
Running command: python3 RDFMUTATE/execute_benchmark.py --config config.yaml

Running RDFMutate Generator...
JAR: rdfmutate-1.1.1.jar
Config: config.yaml
Command: java -jar /home/diego/Documents/benchmarks-synthetic-data-generators/RDFMUTATE/rdfmutate-1.1.1.jar --config=/home/diego/Documents/benchmarks-synthetic-data-generators/RDFMUTATE/temp_config.yaml

✅ RDFMutate Generation Complete!
⏱️  Execution Time: 3.998 seconds
🔄 Mutations Applied: 200
➕ Axioms Added: 33
➖ Axioms Deleted: 0
📁 Output Location: /home/diego/Documents/benchmarks-synthetic-data-generators/RDFMUTATE/output
🎯 Affected Nodes: [frank,grace,company1,alice,eve,diana,bob,company3,henry,charlie,company2]

📊 Detailed report saved to: /home/diego/Documents/benchmarks-synthetic-data-generators/RDFMUTATE/output/benchmark_report.json

✨ Benchmark execution completed successfully!

✅ RDFMutate benchmark completed successfully!


## 7. Execute RDFGraphGen Benchmark

In [9]:
print("=" * 80)
print("EXECUTING RDFGRAPHGEN BENCHMARK")
print("=" * 80)

rdfgraphgen_cmd = [
    "python3",
    "RDFGRAPHGEN/execute_benchmark.py",
    "--shape", RDFGRAPHGEN_SHAPE,
    "--output", RDFGRAPHGEN_OUTPUT,
    "--scale-factor", str(RDFGRAPHGEN_SCALE_FACTOR)
]

print(f"Running command: {' '.join(rdfgraphgen_cmd)}\n")

rdfgraphgen_result = subprocess.run(
    rdfgraphgen_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(rdfgraphgen_result.stdout)
if rdfgraphgen_result.returncode != 0:
    print("ERROR:", rdfgraphgen_result.stderr)
else:
    print("✅ RDFGraphGen benchmark completed successfully!")

EXECUTING RDFGRAPHGEN BENCHMARK
Running command: python3 RDFGRAPHGEN/execute_benchmark.py --shape input-shape.ttl --output output-graph.ttl --scale-factor 100

Running RDFGraphGen...
Shape file: input-shape.ttl
Output file: /home/diego/Documents/benchmarks-synthetic-data-generators/RDFGRAPHGEN/output/output-graph.ttl
Scale factor: 100
Command: rdfgen /home/diego/Documents/benchmarks-synthetic-data-generators/RDFGRAPHGEN/input-shape.ttl /home/diego/Documents/benchmarks-synthetic-data-generators/RDFGRAPHGEN/output/output-graph.ttl 100

✅ RDFGraphGen Generation Complete!
⏱️  Execution Time: 2.676 seconds
📈 Scale Factor: 100
🔢 Total Triples: 3,657
⚡ Triples/sec: 1,367
📁 Output File: /home/diego/Documents/benchmarks-synthetic-data-generators/RDFGRAPHGEN/output/output-graph.ttl
💾 File Size: 0.13 MB

📊 Detailed report saved to: /home/diego/Documents/benchmarks-synthetic-data-generators/RDFGRAPHGEN/output/benchmark_report.json

✨ Benchmark execution completed successfully!

✅ RDFGraphGen bench

## 8. Execute RUDOF Generate Benchmark

In [4]:
print("=" * 80)
print("EXECUTING RUDOF GENERATE BENCHMARK (BINARY v0.1.142)")
print("=" * 80)

rudofgenerate_cmd = [
    "python3",
    "RUDOFGENERATE/execute_benchmark_binary.py",
    "--schema", RUDOFGENERATE_SCHEMA,
    "--entity-count", str(RUDOFGENERATE_ENTITY_COUNT),
    "--output-dir", "output",
    "--output-format", RUDOFGENERATE_OUTPUT_FORMAT,
    "--seed", str(RUDOFGENERATE_SEED),
    "--parallel-threads", str(RUDOFGENERATE_PARALLEL_THREADS)
]

print(f"Running command: {' '.join(rudofgenerate_cmd)}\n")

rudofgenerate_result = subprocess.run(
    rudofgenerate_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(rudofgenerate_result.stdout)
if rudofgenerate_result.returncode != 0:
    print("ERROR:", rudofgenerate_result.stderr)
else:
    print("✅ RUDOF Generate (Binary) benchmark completed successfully!")

EXECUTING RUDOF GENERATE BENCHMARK (BINARY v0.1.142)
Running command: python3 RUDOFGENERATE/execute_benchmark_binary.py --schema example_schema.shex --entity-count 100000 --output-dir output --output-format turtle --seed 42 --parallel-threads 8


RUDOF Generate Binary Benchmark
Schema: /home/diego/Documents/benchmarks-synthetic-data-generators/RUDOFGENERATE/example_schema.shex
Output: /home/diego/Documents/benchmarks-synthetic-data-generators/RUDOFGENERATE/output/generated_data.ttl
Entity count: 100000
Output format: turtle
Seed: 42
Parallel threads: 8

Running command: rudof generate --schema /home/diego/Documents/benchmarks-synthetic-data-generators/RUDOFGENERATE/example_schema.shex --entities 100000 --output-file /home/diego/Documents/benchmarks-synthetic-data-generators/RUDOFGENERATE/output/generated_data.ttl --result-format turtle --seed 42 --parallel 8
RUDOF stdout: 
RUDOF stderr:  INFO /home/diego/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/data_generator-0.1.118/src/pa

## 7. Load Benchmark Reports

In [10]:
# Load reports from the benchmarks we just ran
bsbm_report_path = Path("BSBM/output/benchmark_report.json")
with open(bsbm_report_path, 'r') as f:
    bsbm_report = json.load(f)

lubm_report_path = Path("LUBM/output/benchmark_report.json")
with open(lubm_report_path, 'r') as f:
    lubm_report = json.load(f)

linkgen_report_path = Path("LINKGEN/output/benchmark_report.json")
with open(linkgen_report_path, 'r') as f:
    linkgen_report = json.load(f)

rudofgenerate_report_path = Path("RUDOFGENERATE/output/benchmark_report.json")
with open(rudofgenerate_report_path, 'r') as f:
    rudofgenerate_report = json.load(f)

print("✅ Fresh reports loaded successfully")
print()

# Extract data with different structures
print("📊 BENCHMARK COMPARISON RESULTS")
print("=" * 60)

# BSBM
bsbm_triples = bsbm_report['generated_data']['triples_total']
bsbm_time = bsbm_report['execution']['time_seconds']
bsbm_rate = bsbm_report['generated_data']['triples_per_second']
print(f"🔵 BSBM: {bsbm_triples:,} triples in {bsbm_time:.2f}s ({bsbm_rate:,.0f} triples/sec)")

# LUBM (different structure)
lubm_triples = lubm_report['triples_generated']['total_triples']
lubm_time = lubm_report['execution']['time_seconds']
lubm_rate = lubm_report['performance_metrics']['triples_per_second']
print(f"🔴 LUBM: {lubm_triples:,} triples in {lubm_time:.2f}s ({lubm_rate:,.0f} triples/sec)")

# LINKGEN (different structure again)
linkgen_triples = linkgen_report['generated_data']['triples_requested']
linkgen_time = linkgen_report['execution']['time_seconds']
linkgen_rate = linkgen_report['performance_metrics']['triples_per_second']
print(f"🟢 LINKGEN: {linkgen_triples:,} triples in {linkgen_time:.2f}s ({linkgen_rate:,.0f} triples/sec)")

# RUDOF Generate Binary
rudof_triples = rudofgenerate_report['generated_data']['triples_total']
rudof_time = rudofgenerate_report['execution']['time_seconds']
rudof_rate = rudofgenerate_report['performance_metrics']['triples_per_second']
print(f"🟠 RUDOF Generate (Binary v0.1.142): {rudof_triples:,} triples in {rudof_time:.2f}s ({rudof_rate:,.0f} triples/sec)")

print("=" * 60)
print()
print("🏆 PERFORMANCE RANKING (by triples/second):")
results = [
    ("BSBM", bsbm_rate),
    ("LUBM", lubm_rate),
    ("LINKGEN", linkgen_rate),
    ("RUDOF Generate (Binary)", rudof_rate)
]
results.sort(key=lambda x: x[1], reverse=True)

for i, (name, rate) in enumerate(results, 1):
    medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{medal} {i}. {name}: {rate:,.0f} triples/sec")

✅ Fresh reports loaded successfully

📊 BENCHMARK COMPARISON RESULTS
🔵 BSBM: 3,564,773 triples in 7.91s (450,564 triples/sec)
🔴 LUBM: 1,316,322 triples in 4.34s (303,595 triples/sec)
🟢 LINKGEN: 2,000,000 triples in 99.59s (20,082 triples/sec)
🟠 RUDOF Generate (Binary v0.1.142): 149,999 triples in 24.45s (6,134 triples/sec)

🏆 PERFORMANCE RANKING (by triples/second):
🥇 1. BSBM: 450,564 triples/sec
🥈 2. LUBM: 303,595 triples/sec
🥉 3. LINKGEN: 20,082 triples/sec
   4. RUDOF Generate (Binary): 6,134 triples/sec


In [ ]:
# Load RDFGraphGen report
rdfgraphgen_report_path = Path("RDFGRAPHGEN/output/benchmark_report.json")
with open(rdfgraphgen_report_path, 'r') as f:
    rdfgraphgen_report = json.load(f)

print("✅ RDFGraphGen report loaded successfully")

In [ ]:
# Load RUDOF Generate report
rudofgenerate_report_path = Path("RUDOFGENERATE/output/benchmark_report.json")
with open(rudofgenerate_report_path, 'r') as f:
    rudofgenerate_report = json.load(f)

print("✅ RUDOF Generate report loaded successfully")

## 8. Performance Comparison

In [ ]:
# Create comparison dataframe
import re

# Build comparison data with safe key access
def safe_get(data, keys, default=0):
    """Safely get nested dict value"""
    try:
        result = data
        for key in keys:
            result = result[key]
        return result
    except (KeyError, TypeError):
        return default

# Debug: Check report structures
print("🔍 Debugging report structures...")
print("BSBM keys:", list(bsbm_report.keys()))
if 'generated_data' in bsbm_report:
    print("BSBM generated_data keys:", list(bsbm_report['generated_data'].keys()))

print("LUBM keys:", list(lubm_report.keys()))
if 'file_info' in lubm_report:
    print("LUBM file_info keys:", list(lubm_report['file_info'].keys()))

print("GAIA keys:", list(gaia_report.keys()))
if 'results' in gaia_report:
    print("GAIA results keys:", list(gaia_report['results'].keys()))

print("LINKGEN keys:", list(linkgen_report.keys()))
if 'generated_data' in linkgen_report:
    print("LINKGEN generated_data keys:", list(linkgen_report['generated_data'].keys()))

print("RDFMutate keys:", list(rdfmutate_report.keys()))
if 'mutations' in rdfmutate_report:
    print("RDFMutate mutations keys:", list(rdfmutate_report['mutations'].keys()))
print()

# Extract GAIA instances count from stdout (for reference)
def extract_gaia_instances(stdout_text):
    """Extract number of instances from GAIA stdout"""
    try:
        lines = stdout_text.split('\n')
        for line in lines:
            if 'Done!' in line and 'instances has been generated' in line:
                numbers = re.findall(r'\d+', line)
                if numbers:
                    return int(numbers[0])
            elif 'instances writted' in line:
                numbers = re.findall(r'\d+', line)
                if numbers:
                    return int(numbers[0])
        return 0
    except Exception as e:
        print(f"Warning: Could not extract GAIA instances: {e}")
        return 0

# Count actual triples in GAIA OWL file
def count_gaia_triples(owl_file_path):
    """Count actual RDF triples in the GAIA generated OWL file"""
    try:
        with open(owl_file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Count different types of RDF statements
        # Individual declarations (instances)
        individuals = len(re.findall(r'<[^>]+\s+rdf:about=', content))
        
        # Property assertions (triples with properties)
        property_assertions = len(re.findall(r'<[^>]+:[^>]+>', content)) - individuals
        
        # Type assertions (rdf:type statements)
        type_assertions = len(re.findall(r'rdf:type', content))
        
        # Simple estimate: each individual typically generates multiple triples
        # (type assertion + property assertions)
        estimated_triples = individuals * 3 + property_assertions  # Conservative estimate
        
        print(f"📊 GAIA OWL Analysis:")
        print(f"   • Individual declarations: {individuals}")
        print(f"   • Property assertions: {property_assertions}")
        print(f"   • Type assertions: {type_assertions}")
        print(f"   • Estimated total triples: {estimated_triples}")
        
        return estimated_triples
        
    except Exception as e:
        print(f"Warning: Could not count GAIA triples: {e}")
        # Fallback: use a multiplier on instances
        instances = extract_gaia_instances(gaia_report['stdout'])
        return instances * 4  # Average of 4 triples per instance

gaia_instances = extract_gaia_instances(gaia_report['stdout'])
gaia_triples = count_gaia_triples(Path("GAIA/output/gaia_instances.owl"))

print(f"📊 GAIA Summary:")
print(f"   • Instances generated: {gaia_instances}")
print(f"   • Triples generated: {gaia_triples}")
print()

# Count RDFMutate triples from output file
def count_rdfmutate_triples(ttl_file_path):
    """Count triples in RDFMutate output TTL file"""
    try:
        with open(ttl_file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        # Count lines ending with period (triples in TTL format)
        triple_lines = [line for line in content.split('\n') if line.strip() and not line.strip().startswith('@') and not line.strip().startswith('PREFIX')]
        # Filter out empty lines and count actual triples (lines with content)
        triples = len([line for line in triple_lines if '.' in line or line.strip()])
        return triples
    except Exception as e:
        print(f"Warning: Could not count RDFMutate triples: {e}")
        # Fallback to reported mutations
        return safe_get(rdfmutate_report, ['mutations', 'net_change'], 1)

rdfmutate_triples = count_rdfmutate_triples(Path("RDFMUTATE/output/mutated.ttl"))

print(f"📊 RDFMutate Summary:")
print(f"   • Mutations applied: {safe_get(rdfmutate_report, ['mutations', 'total_mutations'])}")
print(f"   • Axioms added: {safe_get(rdfmutate_report, ['mutations', 'axioms_added'])}")
print(f"   • Total triples: {rdfmutate_triples}")
print()

comparison_data = {
    'Benchmark': ['BSBM', 'LUBM', 'GAIA', 'LINKGEN', 'PyGraft', 'RDFMutate', 'RDFGraphGen', 'RUDOF Generate'],
    'Execution Time (s)': [
        safe_get(bsbm_report, ['execution', 'time_seconds']),
        safe_get(lubm_report, ['execution', 'time_seconds']),
        safe_get(gaia_report, ['results', 'execution_time_seconds']),
        safe_get(linkgen_report, ['execution', 'time_seconds']),
        safe_get(pygraft_report, ['execution', 'time_seconds']),
        safe_get(rdfmutate_report, ['execution', 'time_seconds']),
        safe_get(rdfgraphgen_report, ['execution', 'time_seconds']),
        safe_get(rudofgenerate_report, ['execution', 'time_seconds'])
    ],
    'Total Triples': [
        safe_get(bsbm_report, ['generated_data', 'triples_total']),
        safe_get(lubm_report, ['triples_generated', 'total_triples']),
        gaia_triples,  # Use actual triples count, not instances
        safe_get(linkgen_report, ['generated_data', 'triples_requested']),  # LINKGEN reports requested triples
        safe_get(pygraft_report, ['generated_data', 'estimated_triples']),
        rdfmutate_triples,  # Use counted triples
        safe_get(rdfgraphgen_report, ['generated_data', 'triples_total']),
        safe_get(rudofgenerate_report, ['generated_data', 'triples_total'])
    ],
    'Output Size (MB)': [
        safe_get(bsbm_report, ['generated_data', 'file_size_mb'], 
                 safe_get(bsbm_report, ['generated_data', 'size_mb'])),  # Try alternate key
        safe_get(lubm_report, ['file_info', 'total_size_mb']),
        safe_get(gaia_report, ['results', 'output_size_mb']),
        safe_get(linkgen_report, ['generated_data', 'total_size_mb']),
        safe_get(pygraft_report, ['generated_data', 'total_size_mb']),
        Path("RDFMUTATE/output/mutated.ttl").stat().st_size / (1024 * 1024) if Path("RDFMUTATE/output/mutated.ttl").exists() else 0,
        safe_get(rdfgraphgen_report, ['generated_data', 'file_size_mb']),
        safe_get(rudofgenerate_report, ['generated_data', 'file_size_mb'])
    ]
}

df_comparison = pd.DataFrame(comparison_data)

# Calculate Triples/Second for display
df_comparison['Triples/Second'] = df_comparison['Total Triples'] / df_comparison['Execution Time (s)']

print("\n📊 Performance Comparison (8 benchmarks):")
print("=" * 140)
print(df_comparison.to_string(index=False))
print("=" * 140)

## 9. Visualization: Execution Time Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Execution Time
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#34495e', '#e67e22']  # Blue, Red, Green, Orange, Purple, Teal, Dark Gray, Carrot Orange
ax1.bar(df_comparison['Benchmark'], df_comparison['Execution Time (s)'], color=colors)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('Execution Time Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(df_comparison['Execution Time (s)']):
    ax1.text(i, v + max(df_comparison['Execution Time (s)']) * 0.02, 
             f'{v:.2f}s', ha='center', va='bottom', fontweight='bold')

# Triples per Second
ax2.bar(df_comparison['Benchmark'], df_comparison['Triples/Second'], color=colors)
ax2.set_ylabel('Triples/Second', fontsize=12)
ax2.set_title('Generation Speed Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(df_comparison['Triples/Second']):
    ax2.text(i, v + max(df_comparison['Triples/Second']) * 0.02, 
             f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Visualization: Total Triples Generated

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#34495e', '#e67e22']  # 8 colors
bars = ax.bar(df_comparison['Benchmark'], df_comparison['Total Triples'], 
              color=colors, edgecolor='black', linewidth=1.2)

ax.set_ylabel('Number of Triples', fontsize=12)
ax.set_title('Total Triples Generated Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(df_comparison['Total Triples']):
    ax.text(i, v + max(df_comparison['Total Triples']) * 0.02, 
            f'{v:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Visualization: Output Size Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#34495e', '#e67e22']  # 8 colors
bars = ax.bar(df_comparison['Benchmark'], df_comparison['Output Size (MB)'], 
              color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

ax.set_ylabel('File Size (MB)', fontsize=12)
ax.set_title('Output File Size Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(df_comparison['Output Size (MB)']):
    ax.text(i, v + max(df_comparison['Output Size (MB)']) * 0.02, 
            f'{v:.2f} MB', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 12. Detailed Analysis & Summary

In [ ]:
print("=" * 80)
print("EXECUTING RDFMUTATE BENCHMARK")
print("=" * 80)

rdfmutate_cmd = [
    "python3",
    "RDFMUTATE/execute_benchmark.py",
    "--config", RDFMUTATE_CONFIG,
    "--mutations", str(RDFMUTATE_MUTATIONS),
    "--mutants", str(RDFMUTATE_MUTANTS)
]

print(f"Running command: {' '.join(rdfmutate_cmd)}\n")

rdfmutate_result = subprocess.run(
    rdfmutate_cmd,
    capture_output=True,
    text=True,
    cwd=Path.cwd()
)

print(rdfmutate_result.stdout)
if rdfmutate_result.returncode != 0:
    print("ERROR:", rdfmutate_result.stderr)
else:
    print("✅ RDFMutate benchmark completed successfully!")